# 🗄️ **Nodata Management**

> ### 📌 **TL;DR**
>
> This Jupyter notebook has been created to compare different features with several open source Python libraries for rasters management.
>
> This notebook focuses on nodata and missing-value management in rasters across several libraries.</br>
> It compares how each library represents, propagates, and modifies nodata values during common operations.
>
> The following libraries will be considered :
>- `rasterio`
>- `rioxarray`
>- `odc-geo`
>- `geoutils`

In [ ]:
import numpy as np

import rasterio
from rasterio.fill import fillnodata
from rasterio.plot import show

import rioxarray

import odc.geo.xr

import geoutils as gu

import matplotlib.pyplot as plt

import seaborn as sns

In [ ]:
#path to the raster object with missing values
raster_path = "../data/rasters/subset_nodata.tif"

## **Nodata handling**

In this example we will work on a raster image that has several missing rows and see how different libraries are dealing with nodata values.

In [ ]:
with rasterio.open(raster_path) as src:
    show(src, cmap="mako")

### • *rasterio*

With `rasterio`, we can create numpy masked arrays to handle missing values.

When creating the array, we need to pass the argument `masked=True` when using the `.read()` method on our dataset, as follows:

In [ ]:
with rasterio.open(raster_path) as src:
    arr_masked = src.read(1, masked=True)
    show(arr_masked, cmap="mako")

The masked array created is actually composed of two arrays : one with the actual values (with `--` for missing data), and one with booleans.

In [ ]:
arr_masked

We can use `.mask` to access the boolean matrix (`True` if the pixel has nodata):

In [ ]:
print(arr_masked.mask)

And `.data` allows to check the actual values of the array (with NaNs as missing values):

In [ ]:
print(arr_masked.data)

### • *rioxarray*

Using `rioxarray`, we can convert missing values to NaNs as follows:

In [ ]:
da_rxr = rioxarray.open_rasterio(raster_path)

In [ ]:
da_rxr.sel(band=1).plot.imshow(cmap="mako")
plt.show()

In [ ]:
da_rxr.sel(band=1).values

We can filter missing values with `.where()`:

In [ ]:
da_rxr = da_rxr.where(da_rxr != 0)

In [ ]:
da_rxr.sel(band=1).values

The second option to handle missing values is to open the raster with the argument `mask_and_scale=True` as follows:

In [ ]:
da_rxr_masked = rioxarray.open_rasterio(raster_path, mask_and_scale=True)

In [ ]:
da_rxr_masked.sel(band=1).values

As we can see, this works only if the array contains nodata information stored on disk, telling rioxarray that 0 is a nodata value. Otherwise, this won't work and the nodata has to be set afterwards.

In [ ]:
da_rxr_masked.rio.write_nodata(0, inplace=True)
da_rxr_masked

### • *odc-geo*

`odc-geo` allows conversion to float + NaNs.

In [ ]:
ds_odcgeo = rioxarray.open_rasterio(raster_path)

In [ ]:
ds_odcgeo.sel(band=1).plot.imshow(cmap="mako")
plt.show()

As we are working with an `xarray` structure, we can mimic what we did with `rioxarray` and use `.where()` to filter the missing values by NaN:

In [ ]:
ds_odcgeo = ds_odcgeo.where(ds_odcgeo != 0)
ds_odcgeo.sel(band=1)

In [ ]:
ds_odcgeo.sel(band=1).data

### • *geoutils*

`geoutils` has a function [`set_nodata()`](https://geoutils.readthedocs.io/en/stable/gen_modules/geoutils.Raster.set_nodata.html#geoutils.Raster.set_nodata) to handle missing values.

This function will set a new nodata value for all bands.

In [ ]:
gu_rast = gu.Raster(raster_path)

In [ ]:
gu_rast.plot(bands=1, cmap="mako")

In [ ]:
gu_rast

In [ ]:
gu_rast.nodata

`geoutils` both manages masked arrays and NaNs.

By default, when instantiating the Raster, the object is "lazy", meaning that it is not yet loaded into memory. However when accessing the array with `.data` it will load the data in memory.

If using `.data` to access values, it will return a masked array (with `--` to symbolize nodata values):

In [ ]:
gu_rast[0].data

Using `.mask` will return the boolean mask of the precedent array, with `True` where data is masked and `False` where data is valid:

In [ ]:
gu_rast[0].mask

`geoutils` can also handle nan for nodata values, [`.get_nanarray()`](https://geoutils.readthedocs.io/en/stable/gen_modules/geoutils.Raster.get_nanarray.html#geoutils.Raster.get_nanarray) will return a numpy array with NaNs:

In [ ]:
gu_rast.get_nanarray()[0]

## **NaNs VS Masked Arrays**

- *NaNs*

Working with NaN values has several advantages, they are easily handled by most libraries (`pandas`, `xarray`...). They are very useful for fast data manipulation (i.e. `np.isnan()` to return a boolean matrix), and they have a compact format (a single array, no separate mask structure). However working with NaNs with can be a bit tricky, as the data is forced to be casted to float (NaNs are float for `numpy`) even if the input raster uses integers, which can lead to performance or storage issues. This is mitigated by lazy behaviors.
<br>
<br>

- *Masked Arrays*

On the opposite, masked arrays are ideal when we don't want to convert data to float, they work well with integers, unlike NaNs. There is also a clear distinction between the "actual value" and the "mask", whereas NaNs can sometimes be confusing (NaN = nodata or NaN = invalid result ?). On the other hand, masked arrays have a more complex structure than a simple NaN array, less universal and require more storage space than NaNs. Moreover, numpy support of masked arrays is not complete, some functions are missing or less optimized with respect to the default API.

## **Nodata interpolation**

Nodata interpolation is only available in `rasterio` and `rioxarray` libraries.

This method is used to fill gaps where data is missing, to produce cleaner visualizations. Different methods can be used to perform the interpolation : linear, nearest, cubic...

Once again we will work on a raster image that has several missing rows.

In [ ]:
with rasterio.open(raster_path) as src:
    show(src, cmap="mako")

### • *rasterio*

With rasterio, we will use the [`.fillnodata()`](https://rasterio.readthedocs.io/en/latest/api/rasterio.fill.html) function to perform the interpolation.

This function will search for all nodata values of our dataset and use an algorithm to "fill holes" by using the values of valid neighbour pixels.

In [ ]:
ds_rasterio = rasterio.open(raster_path)
    
b1 = ds_rasterio.read(1) #first band

Here we can see that our dataset has several NaN values :

In [ ]:
b1

If we try to use the `fillnodata` function directly, we get an error message, rasterio is asking for a mask for where to apply the interpolation:

In [ ]:
ds_nodata_interp = fillnodata(b1)

So we will create a mask containing all the nodata values with `np.isnan()`:

In [ ]:
mask_nodata = np.isnan(b1)

It will create a boolean array, True if the pixel has no data and False the opposite.

In [ ]:
mask_nodata

`rasterio` will try to interpolate all pixels with the value 0 (or False). Since in our case we have the opposite (missing values marked as True (or 1)), we need to reverse our array with the function `np.invert()` (or directly with `~`):

In [ ]:
mask_nodata = np.invert(mask_nodata)

In [ ]:
mask_nodata

Our mask is now ready to be used with the `.fillnodata()` function:

In [ ]:
ds_nodata_interp = fillnodata(b1, mask=mask_nodata)

If we check our new array, NaN values have been correctly interpolated:

In [ ]:
ds_nodata_interp

And looking at the map, missing data have been well replaced:

In [ ]:
plt.imshow(ds_nodata_interp, cmap="mako")
plt.show()

### • *rioxarray*

With rioxarray, we will use the [`interpolate_na()`](https://corteva.github.io/rioxarray/stable/rioxarray.html#rioxarray.raster_array.RasterArray.interpolate_na) method to perform nodata interpolation.

In [ ]:
da_rxr = rioxarray.open_rasterio(raster_path)

Once again, we can look at our map with missing data:

In [ ]:
da_rxr.sel(band=1).plot(cmap="mako")
plt.show()

And check the actual values of our array:

In [ ]:
da_rxr.values

With `rioxarray`, this time we don't explicitly need to specify the mask on where to apply the interpolation.

As an argument (optional), we can specify the interpolation method ("linear", "nearest" (by default) or "cubic"), if we want to.

In [ ]:
rxr_interp = da_rxr.rio.interpolate_na(method="linear")

Checking our new array, we can see that missing values have been interpolated:

In [ ]:
rxr_interp.values

We can plot our new raster as well:

In [ ]:
rxr_interp.sel(band=1).plot(cmap="mako", vmin=0)
plt.show()

We can finally compare the nodata interpolation results between `rasterio` and `rioxarray`. We can see the methods are not exactly the same:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fig.suptitle("Interpolation results between rasterio and rioxarray", fontweight="bold")

#rasterio interp
axes[0].imshow(ds_nodata_interp, cmap="mako")
axes[0].set_title("rasterio")
axes[0].axis("off")

#rioxarray interp
axes[1].imshow(rxr_interp.sel(band=1), cmap="mako")
axes[1].set_title("rioxarray")
axes[1].axis("off")

plt.show()